In [1]:
import os

In [2]:
%pwd

'd:\\End_to_End_ML_project_for_Demand_Forecasting_and_Inventory_Optimization\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\End_to_End_ML_project_for_Demand_Forecasting_and_Inventory_Optimization'

In [5]:
import pandas as pd

In [6]:
# Define paths
data_dir = 'artifacts/data_ingestion/data/raw'
files = ['sales.parquet', 'inventory.parquet', 'products.parquet', 'stores.parquet',
          'weather.parquet', 'economics.parquet','promotions','suppliers']

# Load DataFrames
sales_df = pd.read_parquet(os.path.join(data_dir, 'sales.parquet'))
inventory_df = pd.read_parquet(os.path.join(data_dir, 'inventory.parquet'))
products_df = pd.read_parquet(os.path.join(data_dir, 'products.parquet'))
stores_df = pd.read_parquet(os.path.join(data_dir, 'stores.parquet'))
weather_df = pd.read_parquet(os.path.join(data_dir, 'weather.parquet'))
economics_df = pd.read_parquet(os.path.join(data_dir, 'economics.parquet'))
promotions_df = pd.read_parquet(os.path.join(data_dir, 'promotions.parquet'))
suppliers_df = pd.read_parquet(os.path.join(data_dir, 'suppliers.parquet'))

In [7]:
for df_name, df in {
    "sales": sales_df,
    "inventory": inventory_df,
    "products": products_df,
    "stores": stores_df,
    "weather": weather_df,
    "economics": economics_df,
    "promotions": promotions_df,
    "suppliers": suppliers_df
}.items():
    print(f"\n{df_name}")
    print(df.columns.tolist())


sales
['date', 'year', 'month', 'week', 'dayofweek', 'is_weekend', 'product_id', 'category', 'supplier_id', 'unit_cost', 'margin_pct', 'unit_price', 'store_id', 'region', 'store_size_sqft', 'store_type', 'temperature', 'rainfall', 'humidity', 'base_demand', 'units_sold', 'revenue']

inventory
['date', 'product_id', 'store_id', 'stock_on_hand', 'stock_in_transit', 'backorder_qty']

products
['product_id', 'category', 'supplier_id', 'unit_cost', 'margin_pct', 'unit_price']

stores
['store_id', 'region', 'store_size_sqft', 'store_type']

weather
['date', 'temperature', 'rainfall', 'humidity']

economics
['date', 'inflation_rate', 'fuel_price', 'consumer_index']

promotions
['date', 'product_id', 'store_id', 'discount_pct']

suppliers
['supplier_id', 'supplier_rating', 'avg_lead_time', 'lead_time_std']


In [8]:
promotions_df.duplicated(subset=["date", "product_id", "store_id"]).sum()

1672

In [9]:
dups = promotions_df[promotions_df.duplicated(subset=["date", "product_id", "store_id"],keep=False)]

print(dups.sort_values(["date", "product_id", "store_id"]).head(20))

            date  product_id  store_id  discount_pct
10358 2022-01-01          26        10            10
47677 2022-01-01          26        10             5
7213  2022-01-01          52        10            15
28935 2022-01-01          52        10            25
24083 2022-01-02          88        10            15
44700 2022-01-02          88        10            30
36995 2022-01-03          46         9            15
48090 2022-01-03          46         9            30
1867  2022-01-04          10        10            15
38246 2022-01-04          10        10            10
31872 2022-01-04          56         5             5
49772 2022-01-04          56         5            30
35672 2022-01-04          99         5            30
48493 2022-01-04          99         5             5
23328 2022-01-05          83         4            15
33537 2022-01-05          83         4            25
6485  2022-01-06           7         2            15
19462 2022-01-06           7         2        

In [10]:
promotions_clean = (promotions_df.groupby(["date", "product_id", "store_id"], as_index=False).agg({"discount_pct": "max"}))

In [11]:
promotions_clean.duplicated(subset=["date", "product_id", "store_id"]).sum()

0

In [12]:
master_df = sales_df.copy()

# Inventory
master_df = master_df.merge(
    inventory_df,
    on=["date", "product_id", "store_id"],
    how="left",
    validate="one_to_one"
)

# Promotions
master_df = master_df.merge(
    promotions_clean,
    on=["date", "product_id", "store_id"],
    how="left",
    validate="one_to_one"
)

# Supplier Information
master_df = master_df.merge(
    suppliers_df,
    on="supplier_id",
    how="left",
    validate="many_to_one"
)

# Economic Indicators
master_df = master_df.merge(
    economics_df,
    on="date",
    how="left",
    validate="many_to_one"
)

# Fill missing promotions
master_df["discount_pct"] = master_df["discount_pct"].fillna(0)

In [13]:
master_df.duplicated(subset=["date", "product_id", "store_id"]).sum()

0

In [15]:
data = master_df.copy()

In [16]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 730000 entries, 0 to 729999
Data columns (total 32 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   date              730000 non-null  datetime64[ns]
 1   year              730000 non-null  int32         
 2   month             730000 non-null  int32         
 3   week              730000 non-null  int32         
 4   dayofweek         730000 non-null  int32         
 5   is_weekend        730000 non-null  int32         
 6   product_id        730000 non-null  int64         
 7   category          730000 non-null  object        
 8   supplier_id       730000 non-null  int32         
 9   unit_cost         730000 non-null  float64       
 10  margin_pct        730000 non-null  float64       
 11  unit_price        730000 non-null  float64       
 12  store_id          730000 non-null  int64         
 13  region            730000 non-null  object        
 14  stor

In [17]:
data.shape

(730000, 32)

In [18]:
data.isnull().sum()

date                0
year                0
month               0
week                0
dayofweek           0
is_weekend          0
product_id          0
category            0
supplier_id         0
unit_cost           0
margin_pct          0
unit_price          0
store_id            0
region              0
store_size_sqft     0
store_type          0
temperature         0
rainfall            0
humidity            0
base_demand         0
units_sold          0
revenue             0
stock_on_hand       0
stock_in_transit    0
backorder_qty       0
discount_pct        0
supplier_rating     0
avg_lead_time       0
lead_time_std       0
inflation_rate      0
fuel_price          0
consumer_index      0
dtype: int64

In [19]:
data.columns

Index(['date', 'year', 'month', 'week', 'dayofweek', 'is_weekend',
       'product_id', 'category', 'supplier_id', 'unit_cost', 'margin_pct',
       'unit_price', 'store_id', 'region', 'store_size_sqft', 'store_type',
       'temperature', 'rainfall', 'humidity', 'base_demand', 'units_sold',
       'revenue', 'stock_on_hand', 'stock_in_transit', 'backorder_qty',
       'discount_pct', 'supplier_rating', 'avg_lead_time', 'lead_time_std',
       'inflation_rate', 'fuel_price', 'consumer_index'],
      dtype='object')

In [20]:
data.describe()

,date,year,month,week,dayofweek,is_weekend,product_id,supplier_id,unit_cost,margin_pct,...,stock_on_hand,stock_in_transit,backorder_qty,discount_pct,supplier_rating,avg_lead_time,lead_time_std,inflation_rate,fuel_price,consumer_index
count,730000,730000.0,730000.000000,730000.000000,730000.000000,730000.000000,730000.00000,730000.000000,730000.000000,730000.000000,...,730000.000000,730000.000000,730000.000000,730000.000000,730000.000000,730000.000000,730000.000000,730000.000000,730000.000000,730000.000000
mean,2022-12-31 12:00:00,2022.5,6.526027,26.569863,3.006849,0.287671,50.50000,5.310000,277.355617,0.336239,...,1049.761532,249.538422,9.502430,1.167952,4.251941,10.620000,3.080000,5.005527,98.844014,100.035862
min,2022-01-01 00:00:00,2022.0,1.000000,1.000000,0.000000,0.000000,1.00000,1.000000,12.480176,0.204318,...,100.000000,0.000000,0.000000,0.000000,3.587125,3.000000,2.000000,4.121165,69.923677,84.116481
25%,2022-07-02 00:00:00,2022.0,4.000000,14.000000,1.000000,0.000000,25.75000,3.000000,162.538826,0.251981,...,576.000000,124.000000,5.000000,0.000000,3.734028,4.000000,3.000000,4.781577,92.386447,96.731604
50%,2022-12-31 12:00:00,2022.5,7.000000,27.000000,3.000000,0.000000,50.50000,5.500000,293.020086,0.342147,...,1049.000000,250.000000,10.000000,0.000000,4.397988,12.000000,3.000000,5.013684,99.174315,100.074230
75%,2023-07-02 00:00:00,2023.0,10.000000,40.000000,5.000000,1.000000,75.25000,8.000000,400.244965,0.405875,...,1525.000000,375.000000,15.000000,0.000000,4.597991,14.000000,4.000000,5.213814,105.265367,103.359723
max,2023-12-31 00:00:00,2023.0,12.000000,52.000000,6.000000,1.000000,100.00000,10.000000,495.126387,0.492756,...,1999.000000,499.000000,19.000000,30.000000,4.926071,19.000000,4.000000,5.972928,131.520567,115.564551
std,NaN,0.5,3.447854,15.046920,2.001701,0.452677,28.86609,2.958701,141.243631,0.087180,...,548.274754,144.303381,5.766388,4.904240,0.431753,5.403299,0.744044,0.309688,9.913441,4.929685


In [21]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    unzip_data_dir: Path
    STATUS_FILE: Path
    REPORT_FILE: Path
    DRIFT_REPORT_FILE: Path
    STATS_REPORT_FILE: Path
    all_schema: dict
    numerical_ranges: dict
    categorical_values: dict
    thresholds: dict
    high_cardinality_columns: list
    leakage_columns: list
    timestamp_column: str
    target_column: str

In [23]:
from src.demand_forecasting_and_inventory_optimization.constants import *
from src.demand_forecasting_and_inventory_optimization.utils.common import read_yaml, create_directories

In [24]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, schema_filepath=SCHEMA_FILE_PATH, params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)

        self.schema = read_yaml(schema_filepath)

        self.params = read_yaml(params_filepath) 
        
        create_directories(
            [self.config.artifacts_root]
        )

 
    # =====================================================
    # DATA VALIDATION CONFIG
    # =====================================================
    def get_data_validation_config(self) -> DataValidationConfig:

        config = self.config.data_validation

        schema = self.schema

        create_directories(
            [config.root_dir]
        )

        validation_config = (DataValidationConfig(
                root_dir=Path(config.root_dir),

                unzip_data_dir=Path(config.unzip_data_dir),

                STATUS_FILE=Path(config.STATUS_FILE),

                REPORT_FILE=Path(config.REPORT_FILE),

                DRIFT_REPORT_FILE=Path(config.DRIFT_REPORT_FILE),

                STATS_REPORT_FILE=Path(config.STATS_REPORT_FILE),

                all_schema=schema.COLUMNS,

                numerical_ranges=schema.NUMERICAL_RANGES,

                categorical_values=schema.CATEGORICAL_VALUES,

                thresholds=schema.THRESHOLDS,

                high_cardinality_columns=
                schema.HIGH_CARDINALITY_COLUMNS,

                leakage_columns=
                schema.LEAKAGE_COLUMNS,

                timestamp_column=
                schema.TIMESTAMP_COLUMN,

                target_column=
                schema.TARGET_COLUMN
            )
        )

        return validation_config


In [25]:
import os
import json
import warnings

import numpy as np
import pandas as pd

from scipy.stats import ks_2samp

from src.demand_forecasting_and_inventory_optimization import logger
from src.demand_forecasting_and_inventory_optimization.utils.common import save_json

warnings.filterwarnings("ignore")

In [26]:
class DataValidation:

    def __init__(self,config: DataValidationConfig):

        self.config = config

        self.data = self.create_master_dataframe()

        self.validation_report = {}

        self.dataset_statistics = {}

        self.drift_report = {}

        logger.info(
            f"Dataset Loaded Successfully "
            f"Shape={self.data.shape}"
        )

    # =====================================================
    # CREATE MASTER DATAFRAME
    # =====================================================
    def create_master_dataframe(self) -> pd.DataFrame:
        data_dir = self.config.unzip_data_dir
        sales_df = pd.read_parquet(os.path.join(data_dir, 'sales.parquet'))
        inventory_df = pd.read_parquet(os.path.join(data_dir, 'inventory.parquet'))
        products_df = pd.read_parquet(os.path.join(data_dir, 'products.parquet'))
        stores_df = pd.read_parquet(os.path.join(data_dir, 'stores.parquet'))
        weather_df = pd.read_parquet(os.path.join(data_dir, 'weather.parquet'))
        economics_df = pd.read_parquet(os.path.join(data_dir, 'economics.parquet'))
        promotions_df = pd.read_parquet(os.path.join(data_dir, 'promotions.parquet'))
        suppliers_df = pd.read_parquet(os.path.join(data_dir, 'suppliers.parquet'))
        promotions_clean = (promotions_df.groupby(["date", "product_id", "store_id"], as_index=False).agg({"discount_pct": "max"}))
        master_df = sales_df.copy()

        # Inventory
        master_df = master_df.merge(
            inventory_df,
            on=["date", "product_id", "store_id"],
            how="left",
            validate="one_to_one"
        )

        # Promotions
        master_df = master_df.merge(
            promotions_clean,
            on=["date", "product_id", "store_id"],
            how="left",
            validate="one_to_one"
        )

        # Supplier Information
        master_df = master_df.merge(
            suppliers_df,
            on="supplier_id",
            how="left",
            validate="many_to_one"
        )

        # Economic Indicators
        master_df = master_df.merge(
            economics_df,
            on="date",
            how="left",
            validate="many_to_one"
        )

        # Fill missing promotions
        master_df["discount_pct"] = master_df["discount_pct"].fillna(0)
        return master_df
    
    # =====================================================
    # REPORT HELPERS
    # =====================================================

    def _update_report(
        self,
        validation_name: str,
        status: bool,
        details=None
    ):

        self.validation_report[
            validation_name
        ] = {

            "status": status,
            "details": details
        }

    def _save_reports(self):

        save_json(
            self.config.REPORT_FILE,
            self.validation_report
        )

        save_json(
            self.config.STATS_REPORT_FILE,
            self.dataset_statistics
        )

        if len(self.drift_report) > 0:

            save_json(
                self.config.DRIFT_REPORT_FILE,
                self.drift_report
            )

    # =====================================================
    # DATASET STATISTICS
    # =====================================================

    def generate_dataset_statistics(self):

        stats = {

            "rows":
                int(
                    self.data.shape[0]
                ),

            "columns":
                int(
                    self.data.shape[1]
                ),

            "duplicates":
                int(
                    self.data
                    .duplicated()
                    .sum()
                ),

            "memory_usage_mb":
                round(
                    self.data
                    .memory_usage(
                        deep=True
                    )
                    .sum()
                    /
                    1024
                    /
                    1024,
                    2
                ),

            "date_min":
                str(
                    self.data["date"]
                    .min()
                ),

            "date_max":
                str(
                    self.data["date"]
                    .max()
                ),

            "target_stats":
                {
                    "min":
                        float(
                            self.data[
                                self.config.target_column
                            ].min()
                        ),

                    "max":
                        float(
                            self.data[
                                self.config.target_column
                            ].max()
                        ),

                    "mean":
                        float(
                            self.data[
                                self.config.target_column
                            ].mean()
                        ),

                    "std":
                        float(
                            self.data[
                                self.config.target_column
                            ].std()
                        )
                }
        }

        self.dataset_statistics = stats

        logger.info(
            "Dataset Statistics Generated"
        )

    # =====================================================
    # SCHEMA VALIDATION
    # =====================================================

    def validate_schema(self):

        expected_columns = list(
            self.config.all_schema.keys()
        )

        actual_columns = list(
            self.data.columns
        )

        missing_columns = [
            col
            for col in expected_columns
            if col not in actual_columns
        ]

        extra_columns = [
            col
            for col in actual_columns
            if col not in expected_columns
        ]

        validation_passed = (
            len(missing_columns) == 0
        )

        self._update_report(
            "Schema Validation",
            validation_passed,
            {
                "missing_columns":
                    missing_columns,

                "extra_columns":
                    extra_columns
            }
        )

        return validation_passed

    # =====================================================
    # DATATYPE VALIDATION
    # =====================================================

    def validate_datatypes(self):

        mismatches = []

        for (
            col,
            expected_dtype
        ) in self.config.all_schema.items():

            if col not in self.data.columns:
                continue

            actual_dtype = str(
                self.data[col].dtype
            )

            if (
                actual_dtype
                != expected_dtype
            ):

                mismatches.append(
                    {
                        "column":
                            col,

                        "expected":
                            expected_dtype,

                        "actual":
                            actual_dtype
                    }
                )

        validation_passed = (
            len(mismatches) == 0
        )

        self._update_report(
            "Datatype Validation",
            validation_passed,
            mismatches
        )

        return validation_passed

    # =====================================================
    # MISSING VALUES
    # =====================================================

    def validate_missing_values(self):

        threshold = (
            self.config.thresholds
            .missing_value_threshold
        )

        issues = []

        for col in self.data.columns:

            missing_ratio = (
                self.data[col]
                .isnull()
                .mean()
            )

            if (
                missing_ratio
                >
                threshold
            ):

                issues.append(
                    {
                        "column":
                            col,

                        "missing_pct":
                            round(
                                missing_ratio
                                * 100,
                                2
                            )
                    }
                )

        validation_passed = (
            len(issues) == 0
        )

        self._update_report(
            "Missing Value Validation",
            validation_passed,
            issues
        )

        return validation_passed

    # =====================================================
    # DUPLICATES
    # =====================================================

    def validate_duplicates(self):

        duplicate_count = int(
            self.data
            .duplicated()
            .sum()
        )

        duplicate_pct = (
            duplicate_count
            /
            len(self.data)
        )

        validation_passed = (
            duplicate_pct
            <=
            self.config.thresholds
            .max_duplicate_percentage
        )

        self._update_report(
            "Duplicate Validation",
            validation_passed,
            {
                "duplicate_count":
                    duplicate_count,

                "duplicate_pct":
                    round(
                        duplicate_pct * 100,
                        4
                    )
            }
        )

        return validation_passed

    # =====================================================
    # NUMERICAL RANGE
    # =====================================================

    def validate_numerical_ranges(self):

        issues = []

        for (
            col,
            limits
        ) in self.config.numerical_ranges.items():

            if col not in self.data.columns:
                continue

            invalid = self.data[
                (
                    self.data[col]
                    < limits.min
                )
                |
                (
                    self.data[col]
                    > limits.max
                )
            ]

            if len(invalid) > 0:

                issues.append(
                    {
                        "column":
                            col,

                        "violations":
                            int(
                                len(
                                    invalid
                                )
                            )
                    }
                )

        validation_passed = (
            len(issues) == 0
        )

        self._update_report(
            "Numerical Range Validation",
            validation_passed,
            issues
        )

        return validation_passed

    # =====================================================
    # CATEGORICAL VALIDATION
    # =====================================================

    def validate_categorical_values(self):

        issues = []

        for (
            col,
            allowed_values
        ) in (
            self.config
            .categorical_values
            .items()
        ):

            if col not in self.data.columns:
                continue

            invalid = self.data[
                ~self.data[col]
                .isin(
                    list(
                        allowed_values
                    )
                )
            ]

            if len(invalid) > 0:

                issues.append(
                    {
                        "column":
                            col,

                        "invalid_count":
                            len(
                                invalid
                            )
                    }
                )

        validation_passed = (
            len(issues) == 0
        )

        self._update_report(
            "Categorical Validation",
            validation_passed,
            issues
        )

        return validation_passed

    # =====================================================
    # TARGET VALIDATION
    # =====================================================

    def validate_target(self):

        target_col = (
            self.config.target_column
        )

        invalid_count = int(
            (
                self.data[target_col]
                < 0
            ).sum()
        )

        validation_passed = (
            invalid_count == 0
        )

        self._update_report(
            "Target Validation",
            validation_passed,
            {
                "negative_values":
                    invalid_count
            }
        )

        return validation_passed

    # =====================================================
    # TIMESTAMP VALIDATION
    # =====================================================

    def validate_timestamp(self):

        timestamp_col = (
            self.config.timestamp_column
        )

        timestamps = pd.to_datetime(
            self.data[timestamp_col],
            errors="coerce"
        )

        invalid_dates = int(
            timestamps.isnull().sum()
        )

        future_dates = int(
            (
                timestamps
                >
                pd.Timestamp.now()
            ).sum()
        )

        validation_passed = (
            invalid_dates == 0
            and
            future_dates == 0
        )

        self._update_report(
            "Timestamp Validation",
            validation_passed,
            {
                "invalid_dates":
                    invalid_dates,

                "future_dates":
                    future_dates
            }
        )

        return validation_passed

    # =====================================================
    # TIME SERIES CONTINUITY
    # =====================================================

    def validate_time_series_continuity(self):

        dates = pd.to_datetime(
            self.data["date"]
        )

        expected_dates = (
            pd.date_range(
                start=dates.min(),
                end=dates.max(),
                freq="D"
            )
        )

        actual_dates = (
            dates
            .drop_duplicates()
            .sort_values()
        )

        missing_dates = (
            expected_dates
            .difference(
                actual_dates
            )
        )

        validation_passed = (
            len(missing_dates) == 0
        )

        self._update_report(
            "Time Series Continuity",
            validation_passed,
            {
                "missing_dates":
                    len(
                        missing_dates
                    )
            }
        )

        return validation_passed

    # =====================================================
    # BUSINESS RULES
    # =====================================================

    def validate_business_rules(self):

        issues = []

        revenue_check = (
            self.data["units_sold"]
            *
            self.data["unit_price"]
        )

        revenue_mismatch = (
            np.abs(
                revenue_check
                -
                self.data["revenue"]
            )
            >
            0.01
        )

        if revenue_mismatch.sum() > 0:

            issues.append(
                {
                    "rule":
                        "Revenue Consistency",

                    "violations":
                        int(
                            revenue_mismatch
                            .sum()
                        )
                }
            )

        inventory_cols = [

            "stock_on_hand",

            "stock_in_transit",

            "backorder_qty"
        ]

        for col in inventory_cols:

            violations = int(
                (
                    self.data[col]
                    < 0
                ).sum()
            )

            if violations > 0:

                issues.append(
                    {
                        "rule":
                            f"{col} >= 0",

                        "violations":
                            violations
                    }
                )

        validation_passed = (
            len(issues) == 0
        )

        self._update_report(
            "Business Rule Validation",
            validation_passed,
            issues
        )

        return validation_passed

    # =====================================================
    # INVENTORY VALIDATION
    # =====================================================

    def validate_inventory_logic(self):

        stockout_rows = self.data[

            self.data[
                "units_sold"
            ]
            >
            (
                self.data[
                    "stock_on_hand"
                ]
                +
                self.data[
                    "backorder_qty"
                ]
            )
        ]

        self._update_report(
            "Inventory Validation",
            True,
            {
                "potential_stockouts":
                    len(
                        stockout_rows
                    )
            }
        )

        return True

    # =====================================================
    # PRODUCT-STORE COVERAGE
    # =====================================================

    def validate_product_store_coverage(self):

        expected = (
            self.data[
                "product_id"
            ]
            .nunique()
            *
            self.data[
                "store_id"
            ]
            .nunique()
        )

        actual = (
            self.data[
                [
                    "product_id",
                    "store_id"
                ]
            ]
            .drop_duplicates()
            .shape[0]
        )

        validation_passed = (
            actual == expected
        )

        self._update_report(
            "Product Store Coverage",
            validation_passed,
            {
                "expected":
                    expected,

                "actual":
                    actual
            }
        )

        return validation_passed

    # =====================================================
    # DATA FRESHNESS
    # =====================================================

    def validate_data_freshness(self):

        latest_date = (
            pd.to_datetime(
                self.data["date"]
            )
            .max()
        )

        self._update_report(
            "Data Freshness",
            True,
            {
                "latest_date":
                    str(
                        latest_date
                    )
            }
        )

        return True

    # =====================================================
    # INFINITE VALUES
    # =====================================================

    def validate_infinite_values(self):

        issues = []

        numeric_cols = (
            self.data
            .select_dtypes(
                include=np.number
            )
            .columns
        )

        for col in numeric_cols:

            count = int(
                np.isinf(
                    self.data[col]
                ).sum()
            )

            if count > 0:

                issues.append(
                    {
                        "column":
                            col,

                        "count":
                            count
                    }
                )

        validation_passed = (
            len(issues) == 0
        )

        self._update_report(
            "Infinite Value Validation",
            validation_passed,
            issues
        )

        return validation_passed

    # =====================================================
    # OUTLIER ANALYSIS
    # =====================================================

    def validate_outliers(self):

        outlier_report = {}

        numeric_cols = (
            self.data
            .select_dtypes(
                include=np.number
            )
            .columns
        )

        for col in numeric_cols:

            q1 = (
                self.data[col]
                .quantile(0.25)
            )

            q3 = (
                self.data[col]
                .quantile(0.75)
            )

            iqr = q3 - q1

            lower = (
                q1
                -
                1.5 * iqr
            )

            upper = (
                q3
                +
                1.5 * iqr
            )

            outlier_report[col] = int(
                (
                    (
                        self.data[col]
                        < lower
                    )
                    |
                    (
                        self.data[col]
                        > upper
                    )
                ).sum()
            )

        self._update_report(
            "Outlier Validation",
            True,
            outlier_report
        )

        return True

    # =====================================================
    # CARDINALITY
    # =====================================================

    def validate_cardinality(self):

        report = {}

        for col in (
            self.config
            .high_cardinality_columns
        ):

            if col in self.data.columns:

                report[col] = int(
                    self.data[col]
                    .nunique()
                )

        self._update_report(
            "Cardinality Validation",
            True,
            report
        )

        return True

    # =====================================================
    # LEAKAGE CHECK
    # =====================================================

    def validate_data_leakage(self):

        found = []

        dataset_columns = [
            col.lower()
            for col in self.data.columns
        ]

        for leak_col in (
            self.config
            .leakage_columns
        ):

            if (
                leak_col.lower()
                in dataset_columns
            ):

                found.append(
                    leak_col
                )

        validation_passed = (
            len(found) == 0
        )

        self._update_report(
            "Leakage Validation",
            validation_passed,
            found
        )

        return validation_passed

    # =====================================================
    # CORRELATION ANALYSIS
    # =====================================================

    def validate_correlation(self):

        threshold = (
            self.config.thresholds
            .correlation_threshold
        )

        numeric_df = (
            self.data
            .select_dtypes(
                include=np.number
            )
        )

        corr_matrix = (
            numeric_df
            .corr()
            .abs()
        )

        upper = corr_matrix.where(
            np.triu(
                np.ones(
                    corr_matrix.shape
                ),
                k=1
            ).astype(bool)
        )

        high_corr_pairs = []

        for col in upper.columns:

            for row in upper.index:

                corr = upper.loc[
                    row,
                    col
                ]

                if (
                    pd.notnull(corr)
                    and
                    corr > threshold
                ):

                    high_corr_pairs.append(
                        {
                            "feature_1":
                                row,

                            "feature_2":
                                col,

                            "correlation":
                                round(
                                    float(
                                        corr
                                    ),
                                    4
                                )
                        }
                    )

        self._update_report(
            "Correlation Validation",
            True,
            high_corr_pairs
        )

        return True

    # =====================================================
    # DATA DRIFT
    # =====================================================

    def validate_data_drift(
        self,
        train_df,
        test_df
    ):

        threshold = (
            self.config.thresholds
            .drift_pvalue_threshold
        )

        drift_report = {}

        overall_pass = True

        numeric_cols = (
            train_df
            .select_dtypes(
                include=np.number
            )
            .columns
        )

        for col in numeric_cols:

            stat, p_value = ks_2samp(
                train_df[col]
                .dropna(),

                test_df[col]
                .dropna()
            )

            drift = (
                p_value
                <
                threshold
            )

            if drift:

                overall_pass = False

            drift_report[col] = {

                "ks_statistic":
                    float(stat),

                "p_value":
                    float(p_value),

                "drift_detected":
                    drift
            }

        self.drift_report = (
            drift_report
        )

        self._update_report(
            "Data Drift Validation",
            overall_pass,
            drift_report
        )

        return overall_pass

    # =====================================================
    # STATUS FILE
    # =====================================================

    def save_validation_status(
        self,
        validation_results
    ):

        overall_status = all(
            validation_results.values()
        )

        with open(
            self.config.STATUS_FILE,
            "w"
        ) as f:

            f.write(
                "DEMAND FORECASTING DATA VALIDATION\n"
            )

            f.write(
                "=" * 60
            )

            f.write("\n\n")

            for (
                name,
                status
            ) in validation_results.items():

                f.write(
                    f"{name}: "
                    f"{'PASSED' if status else 'FAILED'}\n"
                )

            f.write(
                "\n"
            )

            f.write(
                "=" * 60
            )

            f.write(
                "\nOVERALL STATUS: "
                f"{'PASSED' if overall_status else 'FAILED'}"
            )

    # =====================================================
    # MAIN PIPELINE
    # =====================================================

    def initiate_data_validation(self):

        logger.info(
            "Starting Data Validation"
        )

        self.generate_dataset_statistics()

        validation_results = {

            "Schema Validation":
                self.validate_schema(),

            "Datatype Validation":
                self.validate_datatypes(),

            "Missing Value Validation":
                self.validate_missing_values(),

            "Duplicate Validation":
                self.validate_duplicates(),

            "Numerical Range Validation":
                self.validate_numerical_ranges(),

            "Categorical Validation":
                self.validate_categorical_values(),

            "Target Validation":
                self.validate_target(),

            "Timestamp Validation":
                self.validate_timestamp(),

            "Time Series Continuity":
                self.validate_time_series_continuity(),

            "Business Rule Validation":
                self.validate_business_rules(),

            "Inventory Validation":
                self.validate_inventory_logic(),

            "Product Store Coverage":
                self.validate_product_store_coverage(),

            "Data Freshness":
                self.validate_data_freshness(),

            "Infinite Value Validation":
                self.validate_infinite_values(),

            "Outlier Validation":
                self.validate_outliers(),

            "Cardinality Validation":
                self.validate_cardinality(),

            "Leakage Validation":
                self.validate_data_leakage(),

            "Correlation Validation":
                self.validate_correlation()
        }

        self.save_validation_status(
            validation_results
        )

        self._save_reports()

        overall_status = all(
            validation_results.values()
        )

        logger.info(
            f"Validation Status: "
            f"{overall_status}"
        )

        return overall_status


In [27]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    overall_status = data_validation.initiate_data_validation()
    if overall_status:
        logger.info("Data Validation Completed Successfully")
    else:
        logger.warning("Data Validation Completed with Issues")
except Exception as e:
    raise e

[2026-07-01 13:24:50,413: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-01 13:24:50,446: INFO: common: yaml file: config\schema.yaml loaded successfully]
[2026-07-01 13:24:50,446: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-01 13:24:50,454: INFO: common: created directory at: artifacts]
[2026-07-01 13:24:50,456: INFO: common: created directory at: artifacts/data_validation]
[2026-07-01 13:24:52,422: INFO: 694832464: Dataset Loaded Successfully Shape=(730000, 32)]
[2026-07-01 13:24:52,422: INFO: 694832464: Starting Data Validation]
[2026-07-01 13:24:54,868: INFO: 694832464: Dataset Statistics Generated]
[2026-07-01 13:25:02,993: INFO: common: json file saved at: artifacts\data_validation\validation_report.json]
[2026-07-01 13:25:02,993: INFO: common: json file saved at: artifacts\data_validation\dataset_statistics.json]
[2026-07-01 13:25:03,001: INFO: 694832464: Validation Status: True]
[2026-07-01 13:25:03,001: INFO: 2144668131: Data V